# 04 · Extension: how optimistic is random-split validation for spatio-temporal extremes?

**This notebook is my own extension** of Notebook 01, which comes from the CASE 2025 short course *AI and extremes* (Zhang, Richards & Majumder; [Jbrich95/CASE2025_shortcourse](https://github.com/Jbrich95/CASE2025_shortcourse)). The models, losses and architectures are the course's. The experimental design is new.

## Question

Notebook 01 splits the wildfire data into train, validation and test sets by sampling **individual grid-cell-months at random**. Neighbouring cells in the same month share weather, so a randomly held-out cell usually has close "twins" in the training set. A random split therefore estimates how well the model fills gaps *within months it has already seen*. The question that matters for risk assessment is different: how well does it predict **a period it has not seen?**

## Design (like-for-like)

* **Test set, identical for both schemes:** every positive cell in **2015**. It is never used for training, scaling, early stopping or model choice.
* The two schemes differ **only** in how the 2011–2014 data are split into training and validation:
  * **Random scheme:** cell-months from 2011–2014 assigned to train/validation at random.
  * **Blocked scheme:** train = 2011–2013, validation = 2014, a whole unseen year.
  * Both use the same number of training and validation rows.
* In both schemes, covariates are standardised with **training-row statistics only**. Checkpointing, early stopping and the choice between GPD models 1–3 use the **validation** set only.
* Every $\sigma$/$\xi$ branch is **rebuilt from scratch** for each model, so no model gets a warm start.
* The whole pipeline is repeated over several random seeds, because neural-network training is itself random.

## What is measured

For each scheme: the performance it **predicts** (on its validation set) against the performance it **actually achieves** on 2015.

1. **Pinball loss** of the $\tau = 0.85$ threshold model. This is directly comparable between schemes on the shared 2015 data.
2. **Non-exceedance rate** $P(Y < \hat u(\mathbf x))$, which should equal $\tau = 0.85$.
3. **Mean GPD negative log-likelihood per excess** for the chosen GPD model. Each scheme has its own threshold and therefore its own excess set, so this is compared *within* a scheme (validation vs test), not between schemes.
4. **Skill relative to climatology**, $1 - \text{loss}_{\text{model}} / \text{loss}_{\text{clim}}$, where "climatology" is the covariate-free forecast: the unconditional $\tau$-quantile of the *training* responses. This metric is needed because raw pinball loss scales with the size of burnt areas, and fire activity differs a lot between years. Without it, a quiet validation year makes any model look good.

A scheme whose validation estimate is close to its 2015 performance is giving an honest picture of generalisation.

> **Caveats.** A single validation year (7 months) and a single test year give noisy estimates, and 2015 may simply be an unusual fire year. Real-world use would need multiple held-out years (for example leave-one-year-out cross-validation) and spatial blocking as well as temporal blocking.

In [ ]:
# [Adapted] Setup that works on Google Colab's R runtime and on a local machine.
options(timeout = 1200)                       # large data file on slow connections
on_colab <- dir.exists("/content")
if (on_colab) install.packages(c("keras3", "evd"))
# Local machines: use a Python virtualenv called "r-keras" if one exists
if (!on_colab && reticulate::virtualenv_exists("r-keras")) {
  reticulate::use_virtualenv("r-keras", required = TRUE)
}
library(keras3)

# Course data, downloaded from the original repository if not already present
base_url <- "https://github.com/Jbrich95/CASE2025_shortcourse/raw/main/"
for (f in c("USWild.rda")) {
  if (!file.exists(f)) download.file(paste0(base_url, f), f, mode = "wb")
}
dir.create("checkpoints", showWarnings = FALSE)   # [Adapted] folder used by the checkpoint callbacks

# If the first Keras call fails with a Python/TensorFlow error, run
#   keras3::install_keras()
# then restart the session (Runtime -> Restart session) and run this cell again.

## Settings

Lower `N_SEEDS` to 1 for a quick run. With 3 seeds the notebook takes roughly 30–60 minutes on a CPU.

In [ ]:
N_SEEDS   <- 3        # repeats of the whole pipeline per scheme
TAU       <- 0.85     # threshold quantile level, as in Notebook 01
EPOCHS_U  <- 60
EPOCHS_G  <- 100
PATIENCE  <- 10

## Data (as in Notebook 01) plus calendar-year labels

In [ ]:
load("USWild.rda")
rows <- 127:161
Y <- USWild$BA[rows, , ]
X <- array(dim = c(35, 129, 61, 22))
X[, , , 1] <- USWild$X$X.t2m[rows, , , ]
X[, , , 2] <- USWild$X$X.SPI[rows, , , ]
X[, , , -c(1, 2)] <- USWild$X$X.N[rows, , , 1:20]

X.positive <- apply(X, 4, function(x) x[Y > 0])
Y.positive <- sqrt(Y[Y > 0])

# Calendar year of every positive cell, extracted in the same order as Y[Y > 0]
month.year <- as.integer(substr(USWild$times[rows], 1, 4))
year.positive <- month.year[slice.index(Y, 1)[Y > 0]]

stopifnot(length(year.positive) == length(Y.positive),
          nrow(X.positive) == length(Y.positive))
table(year.positive)

## Index sets for the two schemes

In [ ]:
test.idx <- which(year.positive == 2015)
pool.idx <- which(year.positive <= 2014)

blocked <- list(train = which(year.positive <= 2013),
                val   = which(year.positive == 2014))

make_random <- function(seed) {
  set.seed(seed)
  perm <- sample(pool.idx)
  n_tr <- length(blocked$train)
  list(train = sort(perm[1:n_tr]), val = sort(perm[-(1:n_tr)]))
}

# Integrity checks: no overlap, equal sizes, test year untouched
chk <- function(s) {
  stopifnot(length(intersect(s$train, s$val)) == 0,
            length(intersect(c(s$train, s$val), test.idx)) == 0,
            length(s$train) == length(blocked$train),
            length(s$val) == length(blocked$val))
}
chk(blocked); chk(make_random(1))
cat("train:", length(blocked$train), " val:", length(blocked$val),
    " test (2015):", length(test.idx), "\n")

## Losses (from Notebook 01) and helper functions

In [ ]:
tilted_loss <- function(y_true, y_pred) {
  error <- y_true - y_pred
  op_mean(op_maximum(TAU * error, (TAU - 1) * error))
}

GPD_nll <- function(y_true, y_pred) {
  sigma <- y_pred[all_dims(), 1]
  xi    <- y_pred[all_dims(), 2]
  y     <- y_true[all_dims(), 1]
  -op_sum(-(1 / xi + 1) * op_log(1 + xi * y / sigma) - op_log(sigma))
}

# Plain-R versions, used for evaluation so the numbers do not depend on batching
pinball_mean <- function(y, q) mean(pmax(TAU * (y - q), (TAU - 1) * (y - q)))
gpd_nll_mean <- function(y, sigma, xi) mean(log(sigma) + (1 / xi + 1) * log1p(xi * y / sigma))

ckpt <- function(name) callback_model_checkpoint(
  file.path("checkpoints", paste0(name, ".weights.h5")),
  monitor = "val_loss", save_best_only = TRUE, save_weights_only = TRUE,
  mode = "min", verbose = 0)

### Model builders

These are the same architectures as Notebook 01. Each call builds a **fresh** network.

In [ ]:
build_u <- function() {
  keras_model_sequential(input_shape = 22) %>%
    layer_dense(64, activation = "relu") %>%
    layer_dense(64, activation = "relu") %>%
    layer_dense(1, activation = "exponential") %>%
    compile(optimizer = "adam", loss = tilted_loss)
}

sigma_branch <- function(inp) inp %>%
  layer_dense(32, activation = "sigmoid") %>%
  layer_dense(32, activation = "sigmoid") %>%
  layer_dense(1, activation = "exponential")

build_gpd <- function(type) {
  in_s <- layer_input(shape = 22, name = "input_sigma")
  in_x <- layer_input(shape = 1,  name = "input_xi")
  if (type == "GPD1") {                 # sigma(x), xi(x) from one MLP, both positive
    out <- in_s %>%
      layer_dense(32, activation = "sigmoid") %>%
      layer_dense(32, activation = "sigmoid") %>%
      layer_dense(2, activation = "exponential")
    m <- keras_model(inputs = c(in_s, in_x), outputs = out)
  } else if (type == "GPD2") {          # xi depends on temperature only, in (0, 1)
    xi <- in_x %>%
      layer_dense(16, activation = "relu") %>%
      layer_dense(16, activation = "relu") %>%
      layer_dense(1, activation = "sigmoid")
    m <- keras_model(inputs = c(in_s, in_x),
                     outputs = layer_concatenate(c(sigma_branch(in_s), xi)))
  } else {                              # GPD3: constant xi = exp(w)
    xi <- in_x %>% layer_dense(1, use_bias = FALSE, activation = "exponential")
    m <- keras_model(inputs = c(in_s, in_x),
                     outputs = layer_concatenate(c(sigma_branch(in_s), xi)))
  }
  m %>% compile(optimizer = "adam", loss = GPD_nll)
}

# The second input: temperature (column 1) for GPD2, ones otherwise.
# GPD1 ignores it; it is included so all three models share one interface.
xi_input <- function(type, Xs) {
  if (type == "GPD2") as.matrix(Xs[, 1]) else matrix(1, nrow(Xs), 1)
}

### One complete run of the pipeline

`run_pipeline()` takes train/validation/test indices and returns the validation (estimated) and test (actual) metrics. The test rows are only used **after** all fitting and model selection is finished.

In [ ]:
run_pipeline <- function(split, seed, tag) {
  set.seed(seed); tensorflow::set_random_seed(seed)

  # Standardise with TRAINING rows only
  mu <- colMeans(X.positive[split$train, ])
  sd <- apply(X.positive[split$train, ], 2, sd)
  S  <- function(idx) sweep(sweep(X.positive[idx, , drop = FALSE], 2, mu, "-"), 2, sd, "/")
  Xtr <- S(split$train); Xva <- S(split$val); Xte <- S(test.idx)
  ytr <- Y.positive[split$train]; yva <- Y.positive[split$val]; yte <- Y.positive[test.idx]

  # ---- Stage 1: threshold u(x)
  u <- build_u()
  u %>% fit(Xtr, as.matrix(ytr), validation_data = list(Xva, as.matrix(yva)),
            batch_size = 64, epochs = EPOCHS_U, verbose = 0,
            callbacks = list(ckpt(paste0(tag, "_u"))))
  u <- load_model_weights(u, file.path("checkpoints", paste0(tag, "_u.weights.h5")))
  q_tr <- as.vector(predict(u, Xtr, verbose = 0))
  q_va <- as.vector(predict(u, Xva, verbose = 0))

  # Excesses (train and validation only for now)
  ex <- function(y, q, Xs) { i <- which(y > q); list(y = (y - q)[i], X = Xs[i, , drop = FALSE]) }
  E_tr <- ex(ytr, q_tr, Xtr); E_va <- ex(yva, q_va, Xva)

  # ---- Stage 2: three GPD models, fitted and SELECTED on validation only
  fits <- list(); val_nll <- c()
  for (type in c("GPD1", "GPD2", "GPD3")) {
    g <- build_gpd(type)
    g %>% fit(list(E_tr$X, xi_input(type, E_tr$X)), as.matrix(E_tr$y),
              validation_data = list(list(E_va$X, xi_input(type, E_va$X)), as.matrix(E_va$y)),
              batch_size = 16, epochs = EPOCHS_G, verbose = 0,
              callbacks = list(ckpt(paste0(tag, "_", type)),
                               callback_early_stopping(monitor = "val_loss", patience = PATIENCE)))
    g <- load_model_weights(g, file.path("checkpoints", paste0(tag, "_", type, ".weights.h5")))
    p <- predict(g, list(E_va$X, xi_input(type, E_va$X)), verbose = 0)
    val_nll[type] <- gpd_nll_mean(E_va$y, p[, 1], p[, 2])
    fits[[type]] <- g
  }
  chosen <- names(which.min(val_nll))

  # ---- Only now: evaluate on the untouched 2015 test year
  q_te <- as.vector(predict(u, Xte, verbose = 0))
  E_te <- ex(yte, q_te, Xte)
  p_te <- predict(fits[[chosen]], list(E_te$X, xi_input(chosen, E_te$X)), verbose = 0)

  list(
    summary = data.frame(
      tag = tag, seed = seed, chosen = chosen,
      pinball_val = pinball_mean(yva, q_va), pinball_test = pinball_mean(yte, q_te),
      nonexc_val  = mean(yva < q_va),        nonexc_test  = mean(yte < q_te),
      gpd_nll_val = val_nll[[chosen]],       gpd_nll_test = gpd_nll_mean(E_te$y, p_te[, 1], p_te[, 2]),
      n_exc_val = length(E_va$y), n_exc_test = length(E_te$y)),
    val_nll_all = val_nll,
    qq = list(y = E_te$y, sigma = p_te[, 1], xi = p_te[, 2])
  )
}

## Run both schemes over several seeds

In [ ]:
results <- list()
t0 <- Sys.time()
for (s in seq_len(N_SEEDS)) {
  results[[paste0("blocked_", s)]] <- run_pipeline(blocked, seed = s, tag = paste0("blocked_", s))
  results[[paste0("random_", s)]]  <- run_pipeline(make_random(100 + s), seed = s, tag = paste0("random_", s))
  cat(sprintf("seed %d done (%.1f min elapsed)\n", s, as.numeric(difftime(Sys.time(), t0, units = "mins"))))
}
res <- do.call(rbind, lapply(results, `[[`, "summary"))
res$scheme <- sub("_.*", "", res$tag)
rownames(res) <- NULL
print(res, digits = 4)

# Save straight away, so a later plotting error cannot lose a long run
dir.create("results", showWarnings = FALSE)
write.csv(res, "results/blocked_vs_random_summary.csv", row.names = FALSE)
saveRDS(results, "results/all_runs.rds")

## Results

**Raw optimism** = (estimated on validation) − (actual on 2015). For losses, a *negative* value means the validation set promised a lower loss than the model achieved. For the non-exceedance rate the target is $\tau = 0.85$.

> **Read the raw losses with care.** They are not on a common scale across years. The two cells after this one show why, and give the scale-free comparison.

In [ ]:
res$pinball_optimism <- res$pinball_val - res$pinball_test
res$gpd_optimism     <- res$gpd_nll_val - res$gpd_nll_test

mean_sd <- function(x) sprintf("%.4f (%.4f)", mean(x), if (length(x) > 1) sd(x) else NA)
cols <- c("pinball_val", "pinball_test", "pinball_optimism",
          "nonexc_val", "nonexc_test",
          "gpd_nll_val", "gpd_nll_test", "gpd_optimism")
tab <- sapply(split(res, res$scheme), function(d) sapply(d[cols], mean_sd))
cat("Mean (sd) over", N_SEEDS, "seeds\n")
print(noquote(tab))

cat("\nGPD model chosen on validation:\n")
print(table(res$scheme, res$chosen))

### Why raw losses mislead here: fire activity differs between years

Pinball loss is in the units of $\sqrt{\text{burnt area}}$, so it is larger in big fire years. The table below shows that 2011 and 2012 were much more active than 2013–2015. The blocked scheme validates on 2014, a quiet year, while the random scheme validates on a mix of all four years. The raw "optimism" above therefore mostly reflects *which years* are in each validation set, not how honest the validation is.

In [ ]:
yr_summary <- function(v) c(n = length(v), mean = mean(v), median = median(v),
                            q85 = unname(quantile(v, TAU)), q99 = unname(quantile(v, 0.99)))
print(round(t(sapply(split(Y.positive, year.positive), yr_summary)), 2))

### Skill relative to climatology (scale-free)

The reference forecast is the unconditional $\tau$-quantile of each scheme's **training** responses. It uses no validation or test data, and it is evaluated on exactly the same rows as the model.

In [ ]:
climatology <- function(split) {
  qc <- quantile(Y.positive[split$train], TAU)
  c(clim_val  = pinball_mean(Y.positive[split$val], qc),
    clim_test = pinball_mean(Y.positive[test.idx], qc))
}
clim <- t(sapply(seq_len(nrow(res)), function(i) {
  sp <- if (res$scheme[i] == "blocked") blocked else make_random(100 + res$seed[i])
  climatology(sp)
}))
res[, colnames(clim)] <- clim
res$skill_val      <- 1 - res$pinball_val  / res$clim_val
res$skill_test     <- 1 - res$pinball_test / res$clim_test
res$skill_optimism <- res$skill_val - res$skill_test   # > 0: validation overstated skill

skill_cols <- c("clim_val", "clim_test", "skill_val", "skill_test", "skill_optimism")
cat("Mean (sd) over", N_SEEDS, "seeds\n")
print(noquote(sapply(split(res, res$scheme), function(d) sapply(d[skill_cols], mean_sd))))

b <- res[res$scheme == "blocked", ]; r <- res[res$scheme == "random", ]
b <- b[order(b$seed), ]; r <- r[order(r$seed), ]
d_skill <- r$skill_optimism - b$skill_optimism
cat("\nOverstatement of skill, random minus blocked, per seed:", round(d_skill, 4), "\n")
if (N_SEEDS >= 3) print(t.test(d_skill))
# This test reflects TRAINING randomness only (seeds). There is one test year,
# so it says nothing about variability across years.

**Paired comparison on the shared 2015 test set.** Within each seed, the two threshold models are evaluated on exactly the same 2015 rows, so their test pinball losses can be compared directly. The test asks whether the blocked-trained threshold model is *better* in 2015, not just more honestly validated.

In [ ]:
b <- res[res$scheme == "blocked", ]; r <- res[res$scheme == "random", ]
b <- b[order(b$seed), ]; r <- r[order(r$seed), ]
d <- r$pinball_test - b$pinball_test
cat("2015 test pinball, random minus blocked, per seed:", round(d, 5), "\n")
cat("mean difference:", round(mean(d), 5),
    "  (positive = blocked threshold model did better in 2015)\n")
if (N_SEEDS >= 3) print(t.test(d))
# With only a few seeds, treat this as descriptive. It shows training randomness
# against scheme differences; it is not strong inferential evidence.

### Pooled QQ plots on 2015 (seed 1, chosen GPD model)

In [ ]:
qq_plot <- function(q, main) {
  # evd::pgpd takes one shape value per call, so apply it observation by observation
  u <- mapply(function(y, s, x) evd::pgpd(y, loc = 0, scale = s, shape = x),
              q$y, q$sigma, q$xi)
  e <- qexp(u)
  p <- ppoints(length(e))
  plot(qexp(p), quantile(e, p), pch = 20, asp = 1,
       xlab = "Theoretical Exp(1)", ylab = "Fitted", main = main)
  abline(0, 1, col = "red")
}
options(repr.plot.width = 12, repr.plot.height = 6)
par(mfrow = c(1, 2))
qq_plot(results[["random_1"]]$qq,  "Random scheme: 2015 test excesses")
qq_plot(results[["blocked_1"]]$qq, "Blocked scheme: 2015 test excesses")

## Figures for the README

This saves two PNG files in `figures/`. On Colab, download them from the file panel on the left (folder icon, then `figures/`, then the three dots, then *Download*) and upload them to `my-work/figures/` in the GitHub repository.

In [ ]:
# Figures for the README, saved to figures/ (download them from Colab's file panel)
dir.create("figures", showWarnings = FALSE)
col <- c(blocked = "#0072B2", random = "#D55E00")   # colour-blind-safe pair

headline_figure <- function() {
  layout(matrix(1:2, 1), widths = c(1, 1.25))
  par(mar = c(5.5, 4.5, 4, 1), cex.main = 1.05, las = 1)

  # Panel A: fire activity by year, with the blocked roles marked
  yr_mean <- tapply(Y.positive, year.positive, mean)
  role    <- c("train", "train", "train", "validation", "test")
  bar_col <- c(rep("grey75", 3), col["blocked"], "grey30")
  b <- barplot(yr_mean, col = bar_col, border = NA, ylim = c(0, max(yr_mean) * 1.25),
               ylab = expression("Mean " * sqrt("burnt area")),
               main = "A. Fire activity differs between years")
  text(b, yr_mean, role, pos = 3, cex = 0.8)
  mtext("Labels: roles in the year-blocked split. Raw losses scale with fire activity,",
        side = 1, line = 2.9, cex = 0.75)
  mtext("so they are not comparable across years.", side = 1, line = 3.7, cex = 0.75)

  # Panel B: skill promised by validation vs achieved on 2015
  plot(NA, xlim = c(0.5, 2.5), ylim = c(0, max(res$skill_val) * 1.25), xaxt = "n",
       xlab = "", ylab = "Skill vs climatology (higher = better)",
       main = "B. Skill promised by validation vs achieved in 2015")
  axis(1, at = 1:2, labels = c("Random split", "Year-blocked split"), tick = FALSE)
  for (k in 1:2) {
    s <- c("random", "blocked")[k]
    d <- res[res$scheme == s, ]
    segments(k - 0.12, d$skill_val, k + 0.12, d$skill_test, col = adjustcolor(col[s], 0.35))
    points(rep(k - 0.12, nrow(d)), d$skill_val,  pch = 16, col = adjustcolor(col[s], 0.5))
    points(rep(k + 0.12, nrow(d)), d$skill_test, pch = 17, col = adjustcolor(col[s], 0.5))
    mv <- mean(d$skill_val); mt <- mean(d$skill_test)
    arrows(k - 0.12, mv, k + 0.12, mt, lwd = 3, col = col[s], length = 0.1)
    text(k, max(d$skill_val, d$skill_test) + 0.014, sprintf("overstated by %.3f", mv - mt),
         col = col[s], cex = 0.85)
  }
  legend("bottomleft", c("validation (promised)", "2015 test (achieved)"),
         pch = c(16, 17), bty = "n", cex = 0.8)
  mtext(sprintf("Thin lines: %d seeds; thick arrows: means. One test year.", N_SEEDS),
        side = 1, line = 3, cex = 0.75)
}

png("figures/headline_validation_optimism.png", width = 2200, height = 1050, res = 200)
headline_figure(); invisible(dev.off())

# QQ plots of the 2015 excesses (seed 1, chosen GPD model)
png("figures/qq_2015_excesses.png", width = 2000, height = 1000, res = 200)
par(mfrow = c(1, 2), las = 1)
qq_plot(results[["random_1"]]$qq,  "Random split: 2015 test excesses")
qq_plot(results[["blocked_1"]]$qq, "Year-blocked split: 2015 test excesses")
invisible(dev.off())

# Also show the headline figure in the notebook
options(repr.plot.width = 14, repr.plot.height = 6.5)
headline_figure()
cat("Saved figures/headline_validation_optimism.png and figures/qq_2015_excesses.png\n")

## Software versions

In [ ]:
write.csv(res, "results/blocked_vs_random_summary.csv", row.names = FALSE)   # includes the optimism columns
cat("Saved results/blocked_vs_random_summary.csv\n")
sessionInfo()

## How to read the results

* **Compare skill, not raw loss.** Raw validation-vs-test gaps are dominated by how active each year's fire season was. The climatology skill score removes that scale effect.
* **Overstatement of skill** (`skill_optimism`) is the key quantity: how much better the validation set said the model was than it turned out to be in 2015. A larger value for the **random** scheme means random row-level validation is over-optimistic for this data. That is the point the E5 rainfall project makes when it proposes holding out complete storms, years or ensemble members.
* **Actual 2015 performance** (`pinball_test`, `skill_test`) answers a different question: whether the blocked split also gives a *better model*, or only a more honest estimate. These two can disagree.
* **The non-exceedance rate shows calibration.** Values below 0.85 in 2015 mean the threshold learned on earlier years under-covers the new year, which is **distribution shift**, not overfitting.
* **The skill score was added after seeing the raw results.** It was chosen because the year-scale confound shown above makes raw gaps uninterpretable. Because the choice was made after seeing the data, treat the skill comparison as exploratory and confirm it on new data.
* **Validation scores include selection bias in both schemes.** The validation set is also used for checkpointing, early stopping and choosing the GPD model, so part of each overstatement comes from selection rather than from the split.
* **Different sources of variation.** The random split is redrawn for each seed, while the blocked split is fixed, so the random scheme's spread includes split randomness.
* **What cannot be concluded.** There is a single test year and a single blocked validation year, and the t-tests reflect only training randomness over seeds. Leave-one-year-out cross-validation over all five years, plus spatial blocking, would be needed for a general conclusion. Part of the random scheme's inflated skill may also come from mixing years with very different fire activity, which the covariates can partly explain, rather than purely from near-duplicate neighbouring cells.